In [37]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [38]:
hpi_data = pd.read_excel("../../data/processed_hpi_with_bordering_las.xlsx")
additional_data = pd.read_excel("../../data/additional_data.xlsx")

additional_data['planning_granted_prop'] = additional_data['Total granted; grand total (all)'] / additional_data['Total decisions; grand total (all)']
additional_data['planning_decisions_per_1000'] = additional_data['Total decisions; grand total (all)'] / additional_data['population'] * 1000

additional_data['dwelling_stock_per_1000'] = additional_data['dwelling_stock'] / additional_data['population'] * 1000

In [39]:
additional_data

,LAD25CD,LAD25NM,dwelling_stock,year_month,population,ashe_weekly,base_rate,claimant_count_prop,Total decisions; grand total (all),Total granted; grand total (all),Total refused; grand total (all),rail_station_entry_exit,GDP,CPIH,planning_granted_prop,planning_decisions_per_1000,dwelling_stock_per_1000
0,E06000001,Hartlepool,39310.0,2001-04-01,89811,NaN,5.50,5.2,162,153,5,234942.0,72.0761,1.5,0.944444,1.803788,437.696941
1,E06000001,Hartlepool,39310.0,2001-05-01,89811,NaN,5.25,5,162,153,5,234942.0,72.1460,1.9,0.944444,1.803788,437.696941
2,E06000001,Hartlepool,39310.0,2001-06-01,89811,NaN,5.25,4.9,162,153,5,234942.0,72.3920,1.9,0.944444,1.803788,437.696941
3,E06000001,Hartlepool,39310.0,2001-07-01,90152,NaN,5.25,5,125,121,0,234942.0,72.4063,1.7,0.968000,1.386547,436.041352
4,E06000001,Hartlepool,39310.0,2001-08-01,90152,NaN,5.00,4.7,125,121,0,234942.0,72.5940,2.1,0.968000,1.386547,436.041352
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85243,E09000033,Westminster,132895.0,2024-11-01,209996,947.4,4.75,4.2,1245,1098,147,0.0,101.4706,3.5,0.881928,5.928684,632.845388
85244,E09000033,Westminster,132895.0,2024-12-01,209996,947.4,4.75,4.2,1245,1098,147,0.0,101.9167,3.5,0.881928,5.928684,632.845388
85245,E09000033,Westminster,132895.0,2025-01-01,209996,947.4,4.75,4.2,1236,1084,152,0.0,101.8995,3.9,0.877023,5.885826,632.845388
85246,E09000033,Westminster,132895.0,2025-02-01,209996,947.4,4.50,4.4,1236,1084,152,0.0,102.3612,3.7,0.877023,5.885826,632.845388


In [40]:
full_data = pd.merge(
    hpi_data,
    additional_data, 
    how="left",
    left_on=["Date", "AreaCode"], 
    right_on=["year_month", "LAD25CD"]
)

In [41]:
full_data = full_data.query('Date > "2007-02-01"')

In [42]:
def build_la_pca_embeddings(
    df: pd.DataFrame,
    la_col: str = "AreaCode",
    date_col: str = "Date",
    start_date: str | pd.Timestamp = "2007-03-01",
    end_date: str | pd.Timestamp = "2010-12-31",
    n_components: int = 5,
) -> pd.DataFrame:
    """
    Create LA-level PCA embeddings (fixed per LA) using:
      - population (avg over [start_date, end_date])
      - dwelling_stock (avg over [start_date, end_date])
      - ashe_weekly (avg over [start_date, end_date])
      - area_km2 (snapshot as-of end_date)
      - CoL_distance_km (snapshot as-of end_date)
      - centroid_x, centroid_y (snapshot as-of end_date)

    Returns
    -------
    la_embed : pd.DataFrame
        Index = LA codes, columns = LA_embed_0..LA_embed_{n_components-1}
    """
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])

    start_date = pd.Timestamp(start_date)
    end_date = pd.Timestamp(end_date)

    avg_vars  = ["population", "dwelling_stock", "ashe_weekly"]
    snap_vars = ["area_km2", "CoL_distance_km", "centroid_x", "centroid_y"]

    # ---- 1) Averages over the window (per LA) ----
    wmask = (df[date_col] >= start_date) & (df[date_col] <= end_date)
    la_avg = (
        df.loc[wmask]
        .groupby(la_col)[avg_vars]
        .mean()
    )

    # ---- 2) Snapshots as-of end_date (latest available value up to end_date) ----
    la_snap = (
        df.loc[df[date_col] <= end_date]
        .sort_values(date_col)
        .groupby(la_col)
        .tail(1)
        .set_index(la_col)[snap_vars]
    )

    # ---- 3) Combine into structural matrix ----
    la_struct = la_avg.join(la_snap, how="inner")

    # Drop any LA with missing inputs
    la_struct = la_struct.dropna(axis=0, how="any")

    if la_struct.shape[0] < (n_components + 1):
        raise ValueError(
            f"Not enough LAs with complete structural data to compute "
            f"{n_components} PCA components (have {la_struct.shape[0]})."
        )

    # ---- 4) Scale + PCA ----
    scaler = StandardScaler()
    X = scaler.fit_transform(la_struct.values.astype(float))

    pca = PCA(n_components=n_components, random_state=42)
    Z = pca.fit_transform(X)

    la_embed = pd.DataFrame(
        Z,
        index=la_struct.index,
        columns=[f"LA_embed_{i}" for i in range(n_components)],
    )

    return la_embed

In [43]:
la_embed = build_la_pca_embeddings(full_data, start_date="2007-03-01", end_date="2010-12-31", n_components=5)

la_embed
# Merge back into panel so every (LA, month) row gets the same embedding vector
full_data = full_data.merge(la_embed, left_on="AreaCode", right_index=True, how="left")

In [45]:
full_data.to_excel("../../data/full_data.xlsx", index=False)